# Maia Lite 0.3.0 — Benchmark diagnóstico v2 (420 + 90 paráfrases)

**Avaliação pré-SFT, sem treinamento.** Corrige artigos e concordância, substitui operações matemáticas duplicadas por 90 operações distintas e acrescenta 90 paráfrases factuais pareadas. **Os resultados da v2 não devem ser comparados diretamente aos da v1**, pois o instrumento mudou. Preserve a v1 como referência histórica.

O modelo original v3.8 é carregado com `strict=True`; não há modificação dos pesos. Este notebook não mede perplexidade sem um corpus held-out comprovadamente independente. Atenção a possíveis vazamentos de pré-treinamento, diferenças de comprimento das opções e itens factuais que requerem revisão humana.

In [ ]:
%pip -q install transformers pandas numpy matplotlib tqdm

## 1 — Caminhos, GPU e checkpoint

In [ ]:
from pathlib import Path
import os, json, gc, hashlib, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Fora do Colab: ajuste os caminhos abaixo.')

RUN_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental')
CHECKPOINT_PATH = RUN_DIR / 'latest.pt'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
BENCHMARK_PATH = Path('/content/Maia_Lite_0_3_0_Benchmark_Base_420_v2.jsonl')
PARAPHRASE_PATH = Path('/content/Maia_Lite_0_3_0_Benchmark_Paraphrases_90_v2.jsonl')
OUTPUT_DIR = RUN_DIR / 'evaluation_base_420_v2'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
assert CHECKPOINT_PATH.is_file(), CHECKPOINT_PATH
assert TOKENIZER_DIR.is_dir(), TOKENIZER_DIR
assert BENCHMARK_PATH.is_file(), f'Dataset principal ausente: {BENCHMARK_PATH}'
assert PARAPHRASE_PATH.is_file(), f'Dataset de paráfrases ausente: {PARAPHRASE_PATH}'


## 2 — Arquitetura original do pré-treinamento v3.8 (sem alterações)

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss


## 3 — Carregamento do tokenizer e checkpoint (somente pesos na GPU)

`weights_only=False` é necessário para o checkpoint com estados RNG/otimizador; **execute somente com checkpoint de origem confiável**. A leitura inicial em CPU pode exigir RAM adicional.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), local_files_only=True, use_fast=True, trust_remote_code=False)
assert len(tokenizer) == cfg.vocab_size
assert tokenizer.eos_token_id == 2 and tokenizer.pad_token_id == 1
print('Tokenizer:',len(tokenizer),'EOS:',tokenizer.eos_token_id)

ck = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
step = ck['step']; identity = ck['identity']
model = MaiaGPT(cfg)
model.load_state_dict(ck['model'], strict=True)
del ck
gc.collect()
model.to(DEVICE).eval()
assert model.head.weight.data_ptr() == model.tok.weight.data_ptr()
print('Step:',step,'Parâmetros:',sum(p.numel() for p in model.parameters()))


## 4 — Ler e validar benchmark (420 questões, três idiomas)

In [ ]:
def load_jsonl(path):
    with open(path,encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
main_items=load_jsonl(BENCHMARK_PATH)
para_items=load_jsonl(PARAPHRASE_PATH)
items=main_items+para_items
assert len(main_items)==420 and len(para_items)==90
assert len({x['id'] for x in items})==510
assert {x['language'] for x in main_items}=={'pt','en','es'}
for x in items:
    assert len(x['options'])==3 and len(set(x['options']))==3
    assert 0<=x['correct_index']<3
    assert x['prefix'].strip()
print('Principal:',len(main_items),'Paráfrases:',len(para_items))
print(pd.crosstab(pd.Series([x['category'] for x in main_items],name='categoria'),pd.Series([x['language'] for x in main_items],name='idioma')))


## 5 — Scoring autoregressivo por log-verossimilhança

Usa a probabilidade condicional da **sequência completa de tokens** e registra `logp_sum`, `logp_mean`, `token_count`. A métrica primária é o maior `logp_sum`; `logp_mean` é diagnóstico e não elimina vieses de comprimento. Exclui itens cuja tokenização do prefixo não é preservada. Para comparar alternativas, os logits do prefixo são cacheados apenas logicamente (sem KV cache); não há truncamento silencioso.

In [ ]:
@torch.inference_mode()
def score_choice(prefix, suffix):
    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    full_ids = tokenizer.encode(prefix + suffix, add_special_tokens=False)
    if not prefix_ids or full_ids[:len(prefix_ids)] != prefix_ids:
        return None, 'token_boundary_mismatch'
    if len(full_ids) > cfg.block_size:
        return None, 'context_overflow'
    x = torch.tensor([full_ids[:-1]],dtype=torch.long,device=DEVICE)
    logits,_ = model(x)
    start=len(prefix_ids)-1
    targets=torch.tensor(full_ids[len(prefix_ids):],dtype=torch.long,device=DEVICE)
    lp=F.log_softmax(logits[0,start:start+len(targets)].float(),dim=-1)
    scores=lp.gather(1,targets[:,None]).squeeze(1)
    return {'logp_sum':float(scores.sum().item()),'logp_mean':float(scores.mean().item()),'token_count':len(targets),'first_token_logp':float(scores[0].item())},None

def evaluate_item(item):
    scores=[]; errors=[]
    for suffix in item['options']:
        result,err=score_choice(item['prefix'],suffix)
        scores.append(result);errors.append(err)
    row={k:item[k] for k in ['id','category','language','fact_id','prefix','correct_index','variant']}
    row['split']='paraphrase' if item['variant']=='paraphrase' else 'main'
    row['options']=json.dumps(item['options'],ensure_ascii=False)
    row['errors']=json.dumps(errors,ensure_ascii=False)
    row['valid']=all(s is not None for s in scores)
    if row['valid']:
        lp=[s['logp_sum'] for s in scores]
        mean=[s['logp_mean'] for s in scores]
        best=max(range(3),key=lambda i:lp[i])
        row.update(predicted_index=best,correct=int(best==item['correct_index']),
                   margin_correct_vs_best_wrong=lp[item['correct_index']]-max(lp[i] for i in range(3) if i!=item['correct_index']),
                   top_margin=sorted(lp,reverse=True)[0]-sorted(lp,reverse=True)[1],
                   logp_sum=json.dumps(lp),logp_mean=json.dumps(mean),
                   token_counts=json.dumps([s['token_count'] for s in scores]),
                   first_token_logp=json.dumps([s['first_token_logp'] for s in scores]),
                   predicted_mean_index=max(range(3),key=lambda i:mean[i]))
    return row

# Teste de integridade de scoring
probe=evaluate_item(items[0]);print(probe)


## 6 — Executar 420 questões

A avaliação é sequencial e pode levar tempo; cada item faz três passes. O processo grava resultados parciais a cada 15 questões, permitindo retomar sem repetir os itens concluídos. Para evitar misturar checkpoints diferentes, o nome do arquivo inclui o `step`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
from hashlib import sha256
DATASET_HASH=sha256(BENCHMARK_PATH.read_bytes()+PARAPHRASE_PATH.read_bytes()).hexdigest()
RESULT_CSV=OUTPUT_DIR/f'benchmark_v2_step_{step}_{DATASET_HASH[:12]}.csv'
if RESULT_CSV.exists():
    previous=pd.read_csv(RESULT_CSV)
    assert set(previous['id']).issubset({x['id'] for x in items})
    completed=set(previous['id'].tolist());records=previous.to_dict('records')
    print('Retomando:',len(records),'itens')
else:
    records=[];completed=set()
start=time.time()
for item in tqdm(items,desc='Benchmark v2 510'):
    if item['id'] in completed:continue
    records.append(evaluate_item(item))
    if len(records)%15==0:pd.DataFrame(records).to_csv(RESULT_CSV,index=False)
pd.DataFrame(records).to_csv(RESULT_CSV,index=False)
results=pd.DataFrame(records)
print('Concluídos:',len(results),'Válidos:',int(results['valid'].sum()),'Tempo (s):',round(time.time()-start,1))
print('CSV:',RESULT_CSV)


## 7 — Relatório de acurácia, margens e erros

In [ ]:
valid=results[results['valid']==True].copy()
main=valid[valid['split']=='main'].copy()
para=valid[valid['split']=='paraphrase'].copy()
if main.empty:raise RuntimeError('Nenhum item principal válido.')
summary=main.groupby(['category','language']).agg(n=('correct','size'),accuracy=('correct','mean'),mean_margin=('margin_correct_vs_best_wrong','mean')).reset_index()
print(summary.to_string(index=False))
print('Acurácia principal:',round(main['correct'].mean(),4),'n=',len(main))
print('Acurácia paráfrases:',round(para['correct'].mean(),4),'n=',len(para))
print('Cobertura:',round(len(valid)/len(items),4))
summary.to_csv(OUTPUT_DIR/f'summary_v2_step_{step}.csv',index=False)
# Comparação de scoring: total vs média por token, para revelar sensibilidade a comprimento
main['correct_mean']= (main['predicted_mean_index']==main['correct_index']).astype(int)
print('Acurácia com logP médio/token (diagnóstico):',round(main['correct_mean'].mean(),4))
print('Divergências entre métricas:',int((main['predicted_mean_index']!=main['predicted_index']).sum()))


## 8 — Diagnóstico de sensibilidade e confiabilidade

O conjunto contém um enunciado por fato em cada idioma; portanto a sensibilidade aqui mede **consistência entre idiomas** e não paráfrases dentro do mesmo idioma. Para medir paráfrases, deve-se criar um conjunto adicional de variações mantendo os mesmos fatos e respostas.

In [ ]:
# Robustez: mesmo fato, idioma e alternativas; formulação diferente.
base=main[main['category']=='factual'][['fact_id','language','correct','predicted_index','margin_correct_vs_best_wrong']].rename(columns={'correct':'base_correct','predicted_index':'base_prediction','margin_correct_vs_best_wrong':'base_margin'})
alt=para[['fact_id','language','correct','predicted_index','margin_correct_vs_best_wrong']].rename(columns={'correct':'para_correct','predicted_index':'para_prediction','margin_correct_vs_best_wrong':'para_margin'})
paired=base.merge(alt,on=['fact_id','language'],validate='one_to_one')
paired['both_correct']=(paired['base_correct']==1)&(paired['para_correct']==1)
paired['changed_prediction']=paired['base_prediction']!=paired['para_prediction']
print('Pares válidos:',len(paired))
print('Ambos corretos:',round(paired['both_correct'].mean(),4))
print('Mudou alternativa:',round(paired['changed_prediction'].mean(),4))
print(paired.groupby('language')[['base_correct','para_correct','both_correct','changed_prediction']].mean())
paired.to_csv(OUTPUT_DIR/f'paraphrase_pairs_v2_step_{step}.csv',index=False)
print('\nErros de exemplo:')
print(main.loc[main.correct==0,['id','prefix','options','predicted_index','correct_index']].head(15).to_string(index=False))


## 9 — Perplexidade (opcional, exige corpus held-out comprovadamente independente)

**Não** calculamos perplexidade sobre as próprias questões nem inventamos dados de validação. Defina `HELDOUT_TEXT_PATH` para um `.txt` que não tenha sido usado no pré-treinamento nem no SFT. O cálculo usa blocos não sobrepostos de até 2048 tokens e soma perdas de todos os tokens previstos, exceto o primeiro de cada bloco.

In [ ]:
HELDOUT_TEXT_PATH = None  # exemplo: '/content/heldout_pt.txt'
@torch.inference_mode()
def heldout_perplexity(path, max_tokens=30000):
    text=Path(path).read_text(encoding='utf-8')
    ids=tokenizer.encode(text,add_special_tokens=False)[:max_tokens]
    assert len(ids)>=2
    total_nll=0.0;total_count=0
    for start in tqdm(range(0,len(ids)-1,cfg.block_size),desc='Heldout PPL'):
        chunk=ids[start:start+cfg.block_size]
        if len(chunk)<2:continue
        x=torch.tensor([chunk[:-1]],dtype=torch.long,device=DEVICE)
        y=torch.tensor(chunk[1:],dtype=torch.long,device=DEVICE)
        logits,_=model(x)
        nll=F.cross_entropy(logits[0].float(),y,reduction='sum')
        total_nll+=float(nll.item());total_count+=len(chunk)-1
    return {'tokens_scored':total_count,'mean_nll':total_nll/total_count,'perplexity':float(__import__('math').exp(total_nll/total_count))}
if HELDOUT_TEXT_PATH:
    print(heldout_perplexity(HELDOUT_TEXT_PATH))
else:
    print('Perplexidade não calculada: corpus held-out não fornecido.')


## 10 — Manifesto e comparação futura com SFT

Congele este JSONL e seu SHA-256. Rode o mesmo notebook com pesos SFT, mantendo tokenizer, prompt, opções e regra de scoring. **Não treine com este conjunto**. Não interprete acurácia em 420 itens como medida representativa sem avaliação externa independente.

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
manifest={'project':'Maia Lite 0.3.0','evaluation':'base_420_v2_plus_90_paraphrases','step':int(step),
          'checkpoint_path':str(CHECKPOINT_PATH),'tokenizer_path':str(TOKENIZER_DIR),
          'main_sha256':sha256_file(BENCHMARK_PATH),'paraphrase_sha256':sha256_file(PARAPHRASE_PATH),
          'main_items':len(main_items),'paraphrase_items':len(para_items),
          'scoring':'sum_logp_conditional_full_completion','valid_items':int(results['valid'].sum()),
          'device':str(DEVICE),'torch':torch.__version__,
          'note':'Benchmark v2 diferente da v1; revisão humana e auditoria de sobreposição ainda necessárias; nunca usar para SFT.'}
manifest_path=OUTPUT_DIR/f'manifest_v2_step_{step}.json'
manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(manifest,ensure_ascii=False,indent=2))
print('Relatórios:',OUTPUT_DIR)


## 11 — Intervalos de confiança (bootstrap por grupos)

Reamostra fatos, não linhas, na categoria factual. Em matemática, operações são distintas por idioma. Os intervalos são exploratórios e não substituem validação de independência do corpus.

In [ ]:
import numpy as np
rng=np.random.default_rng(20261116)
for category,group in main.groupby('category'):
    key='fact_id' if category=='factual' else 'id'
    units=[g['correct'].to_numpy(dtype=float) for _,g in group.groupby(key)]
    draws=[]
    for _ in range(2000):
        ix=rng.integers(0,len(units),size=len(units))
        draws.append(np.concatenate([units[i] for i in ix]).mean())
    lo,hi=np.quantile(draws,[0.025,0.975])
    print(f'{category:10s}: {group.correct.mean():.3%} | IC bootstrap 95% [{lo:.3%}, {hi:.3%}] | grupos={len(units)}')
